# 02 — Profit, Loss & Product Retention Analysis
Outputs:
- Profit/Loss by category/region/segment
- Discount vs margin relationship
- Repeat customer rate (retention proxy)
- Product repeat purchase signals

In [1]:
import pandas as pd
import numpy as np

PATH = "data/cleaned/superstore_clean.parquet"
df = pd.read_parquet(PATH)
df.shape, df.head()

((51290, 31),
           category         city        country customer_id     customer_name  \
 0  Office Supplies  Los Angeles  United States   LS-172304  Lycoris Saunders   
 1  Office Supplies  Los Angeles  United States   MV-174854     Mark Van Huff   
 2  Office Supplies  Los Angeles  United States   CS-121304      Chad Sievert   
 3  Office Supplies  Los Angeles  United States   CS-121304      Chad Sievert   
 4  Office Supplies  Los Angeles  United States   AP-109154    Arthur Prichep   
 
    discount market  记录数 order_date        order_id  ... shipping_cost  \
 0       0.0     US    1 2011-01-07  CA-2011-130813  ...          4.37   
 1       0.0     US    1 2011-01-21  CA-2011-148614  ...          0.94   
 2       0.0     US    1 2011-08-05  CA-2011-118962  ...          1.81   
 3       0.0     US    1 2011-08-05  CA-2011-118962  ...          4.59   
 4       0.0     US    1 2011-09-29  CA-2011-146969  ...          1.32   
 
         state sub_category  year        market2 wee

In [2]:
kpis = {
    "revenue": df["sales"].sum(),
    "profit": df["profit"].sum(),
    "loss_abs": df.loc[df["profit"] < 0, "profit"].abs().sum(),
    "orders": df["order_id"].nunique(),
    "customers": df["customer_id"].nunique(),
}
kpis["profit_margin_pct"] = (kpis["profit"] / kpis["revenue"]) * 100
kpis

{'revenue': np.int64(12642905),
 'profit': np.float64(1467457.2912800002),
 'loss_abs': np.float64(920646.15572),
 'orders': 25035,
 'customers': 4873,
 'profit_margin_pct': np.float64(11.606962887722403)}

In [3]:
group_cols = [c for c in ["category","sub_category"] if c in df.columns]
pl_cat = (df.groupby(group_cols, dropna=False)
            .agg(revenue=("sales","sum"),
                 profit=("profit","sum"),
                 loss_abs=("profit", lambda s: s[s<0].abs().sum()),
                 orders=("order_id","nunique"))
            .reset_index())
pl_cat["profit_margin_pct"] = (pl_cat["profit"] / pl_cat["revenue"]) * 100
pl_cat.sort_values("profit", ascending=False).head(15)

,category,sub_category,revenue,profit,loss_abs,orders,profit_margin_pct
14,Technology,Copiers,1509439,258567.54818,71547.49982,2120,17.130043
16,Technology,Phones,1706874,216717.00580,96417.66010,3133,12.696720
0,Furniture,Bookcases,1466559,161924.41950,101446.29730,2284,11.041112
4,Office Supplies,Appliances,1011081,141680.58940,63991.69040,1686,14.012783
1,Furniture,Chairs,1501682,140396.26750,96084.89690,3187,9.349268
13,Technology,Accessories,749307,129626.30620,39857.49820,2889,17.299492
11,Office Supplies,Storage,1127124,108461.48980,76063.97800,4534,9.622853
6,Office Supplies,Binders,461952,72449.84600,52884.06130,5392,15.683414
10,Office Supplies,Paper,244307,59207.68270,10299.49370,3234,24.234951
15,Technology,Machines,779071,58867.87300,78672.74030,1422,7.556163


In [4]:
loss_products = (df[df["profit"] < 0]
                 .groupby(["product_id","product_name"], dropna=False)
                 .agg(loss_abs=("profit", lambda s: s.abs().sum()),
                      revenue=("sales","sum"),
                      orders=("order_id","nunique"))
                 .reset_index()
                 .sort_values("loss_abs", ascending=False))
loss_products.head(20)

,product_id,product_name,loss_abs,revenue,orders
5606,TEC-MA-10000418,Cubify CubeX 3D Printer Double Head Print,9239.9692,6300,2
2597,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,6859.3896,4900,3
5621,TEC-MA-10000822,Lexmark MX611dhe Monochrome Laser Printer,5269.9690,13770,3
2166,OFF-BI-10000545,GBC Ibimaster 500 Manual ProClick Binding System,5098.5660,6697,6
1649,OFF-AP-10001623,"Hoover Stove, White",5071.4430,9463,4
6083,TEC-PH-10002991,"Apple Smart Phone, Full Size",4574.6439,7258,4
5821,TEC-MOT-10003050,"Motorola Smart Phone, Cordless",4493.3520,3278,2
2253,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,4162.0336,4395,4
5763,TEC-MA-10004125,Cubify CubeX 3D Printer Triple Head Print,3839.9904,8000,1
2454,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,3431.6730,2288,2


In [5]:
orders_per_customer = df.groupby("customer_id")["order_id"].nunique()
repeat_customers = (orders_per_customer >= 2).mean() * 100

repeat_customers

np.float64(91.25795198029961)

In [6]:
cohort = (df.groupby(["customer_first_month","months_since_first"])["customer_id"]
            .nunique()
            .reset_index(name="active_customers"))

cohort_pivot = cohort.pivot(index="customer_first_month",
                            columns="months_since_first",
                            values="active_customers").fillna(0)

# convert to retention rate by dividing by cohort size (month 0)
cohort_size = cohort_pivot[0].replace(0, np.nan)
retention = cohort_pivot.divide(cohort_size, axis=0) * 100
retention.round(1).head(12)

months_since_first,0,1,2,3,4,5,6,7,8,9,...,38,39,40,41,42,43,44,45,46,47
customer_first_month,,,,,,,,,,,,,,,,,,,,,
2011-01,100.0,4.3,4.8,5.3,7.2,8.1,6.7,10.0,12.4,6.7,...,10.0,9.6,12.9,17.7,9.6,20.6,19.6,21.5,21.5,18.2
2011-02,100.0,6.5,4.1,5.3,7.1,8.3,6.5,11.2,9.5,10.1,...,13.6,17.8,16.6,11.2,17.2,18.3,15.4,21.3,19.5,0.0
2011-03,100.0,6.5,6.5,11.3,6.9,6.9,9.7,8.5,10.5,10.9,...,18.2,15.0,11.7,16.6,21.5,16.6,20.6,21.5,0.0,0.0
2011-04,100.0,7.1,8.4,7.6,5.3,8.9,6.7,12.0,10.7,4.0,...,18.7,6.7,16.4,23.1,18.7,19.6,18.7,0.0,0.0,0.0
2011-05,100.0,6.9,3.9,10.4,10.0,7.8,16.5,15.6,6.5,4.8,...,10.0,16.9,25.1,16.9,20.8,21.2,0.0,0.0,0.0,0.0
2011-06,100.0,5.5,7.2,10.7,7.8,12.4,13.5,4.3,6.9,6.9,...,13.8,21.9,17.0,21.6,22.8,0.0,0.0,0.0,0.0,0.0
2011-07,100.0,12.3,10.3,4.5,12.3,11.6,8.4,1.9,10.3,7.1,...,20.6,18.1,23.9,20.6,0.0,0.0,0.0,0.0,0.0,0.0
2011-08,100.0,11.3,6.7,10.3,9.2,3.9,6.4,3.5,8.2,8.9,...,15.6,18.1,21.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-09,100.0,9.7,12.4,9.4,3.7,4.4,7.0,7.0,9.4,11.4,...,21.8,20.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
prod_repeat = (df.groupby(["product_id","product_name"])["customer_id"]
                 .nunique()
                 .reset_index(name="unique_customers"))

# customers who bought same product >=2 times
cust_prod_orders = df.groupby(["customer_id","product_id"])["order_id"].nunique().reset_index(name="orders")
repeat_by_prod = (cust_prod_orders[cust_prod_orders["orders"] >= 2]
                  .groupby("product_id")["customer_id"].nunique()
                  .reset_index(name="repeat_customers_for_product"))

prod_ret = prod_repeat.merge(repeat_by_prod, on="product_id", how="left").fillna(0)
prod_ret["product_repeat_rate_pct"] = (prod_ret["repeat_customers_for_product"] / prod_ret["unique_customers"]) * 100

prod_ret.sort_values("product_repeat_rate_pct", ascending=False).head(20)

,product_id,product_name,unique_customers,repeat_customers_for_product,product_repeat_rate_pct
6839,OFF-PA-10003981,"SanDisk Note Cards, Premium",1,1.0,100.000000
5875,OFF-LA-10001246,"Avery Removable Labels, Alphabetical",1,1.0,100.000000
6437,OFF-PA-10001166,Xerox 1932,1,1.0,100.000000
6525,OFF-PA-10001736,"SanDisk Note Cards, 8.5 x 11",2,1.0,50.000000
9254,TEC-CO-10003113,"Hewlett Fax Machine, Color",2,1.0,50.000000
9274,TEC-CO-10003342,"Canon Wireless Fax, High-Speed",2,1.0,50.000000
8921,TEC-BRO-10001568,"Brother Wireless Fax, Digital",2,1.0,50.000000
8552,TEC-AC-10002263,"SanDisk Memory Card, Bluetooth",2,1.0,50.000000
8048,OFF-SU-10003259,"Elite Trimmer, Easy Grip",2,1.0,50.000000
3041,OFF-AR-10000823,Newell 307,3,1.0,33.333333


## Executive Summary (fill after reviewing outputs)
- Overall profit margin: X%
- Loss concentration: top categories/subcategories: ...
- Discount impact: ...
- Repeat customer rate: X%
- Best repeat-driving products: ...
Recommended actions:
1) ...
2) ...
3) ...